# 04 — Model Training (Stage 2)
Run hyperopt + final training + export in a notebook for interactive exploration.
For production / reproducible runs use `scripts/run_training.py` instead.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, '../src')
import pandas as pd
import matplotlib.pyplot as plt

from wdm.config import load_config
from wdm.model.dataset import build_dataset
from wdm.model.tuner import run_hyperopt
from wdm.model.trainer import train_final
from wdm.model.evaluator import evaluate_all

cfg = load_config('bank_marketing')
data = build_dataset(cfg)
print('train/valid/oot shapes:', data.X_train.shape, data.X_valid.shape, data.X_oot.shape)

In [ ]:
best_params, best_loss, trials = run_hyperopt(data.X_train, data.y_train, cfg, max_evals=10)
print('best params:', best_params)
print('best CV PR-AUC:', -best_loss)

In [ ]:
# Plot hyperopt trial loss over time
losses = [t['result']['loss'] for t in trials.trials]
plt.plot(range(1, len(losses) + 1), [-l for l in losses], marker='o')
plt.xlabel('trial')
plt.ylabel('CV PR-AUC')
plt.title('Hyperopt trajectory')
plt.grid(alpha=0.3)

In [ ]:
booster, evals = train_final(best_params, data.X_train, data.y_train, data.X_valid, data.y_valid, cfg)
metrics_df, binned, scores, imp = evaluate_all(booster, data, cfg)
metrics_df